# Глава 15. Свёрточные сети (CNN)

_Ноутбук сгенерирован из учебника «Машинное обучение для начинающих». Запускайте ячейки по порядку сверху вниз в Google Colab._

**Цель главы  🎯**

После этой главы ты сможешь объяснить, как свёртка-фильтр находит узоры на изображении, построить CNN и сравнить её с полносвязной сетью, а также применить transfer learning на готовой модели.

Обычная сеть из прошлой главы разворачивает картинку в длинный ряд чисел через Flatten — и теряет важнейшую информацию о том, где что находится. Для неё пиксель в углу и пиксель в центре просто числа в списке. Свёрточные сети (CNN, Convolutional Neural Networks) устроены умнее: они смотрят на картинку кусочками и понимают её пространственную структуру. Именно CNN стоят за распознаванием лиц, объектов и медицинских снимков.

**Аналогия: скользящий фильтр-рамка  💡**

Представь маленькую рамку-лупу, которую ты двигаешь по всей картинке слева направо, сверху вниз. В каждом положении рамка проверяет один вопрос: «а нет ли здесь вертикальной линии?» или «а нет ли уголка?». Это и есть свёртка (convolution): маленький фильтр скользит по изображению и отмечает, где встретился нужный узор. Один фильтр ищет линии, другой — дуги, третий — пятна.

## Игра с физическим фильтром

Вырежьте из бумаги маленькое окошко 3×3 клетки и поводите им по клетчатому листу с нарисованной фигурой. В каждом положении окошко «видит» только 9 клеток — маленький фрагмент. Так работает свёрточный фильтр: он не смотрит на всю картинку сразу, а изучает её по кусочкам, зато один и тот же фильтр применяется по всей картинке — поэтому CNN узнаёт узор, где бы он ни находился.

## Карты признаков и пулинг

Результат работы фильтра — карта признаков: новая «картинка», показывающая, где на изображении нашёлся искомый узор. Фильтров в слое много, и каждый ищет своё, поэтому карт получается несколько. Затем слой пулинга (pooling) уменьшает карту, оставляя только самое сильное/важное. Это делает сеть компактнее, быстрее и менее чувствительной к мелким сдвигам объекта.

## Иерархия признаков

**Аналогия: от кирпичей к дому  💡**

Первые свёрточные слои находят простые детали: края и линии. Следующие слои складывают из них узоры посложнее: углы, дуги, текстуры. Более глубокие — целые части объекта: глаз, колесо, ухо. А последние — объект целиком. Как из кирпичей собирается стена, из стен — комната, из комнат — дом. Именно эта иерархия делает CNN такими мощными в распознавании образов.

## CNN в коде на CIFAR-10

CIFAR-10 — датасет цветных картинок 32×32: самолёты, кошки, машины, птицы и ещё шесть категорий. Задача сложнее MNIST (цвет, разнообразие), и здесь CNN покажет своё преимущество. Соберём небольшую свёрточную сеть.

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
X_train, X_test = X_train / 255.0, X_test / 255.0

model = Sequential([
    Conv2D(32, (3, 3), activation="relu", input_shape=(32, 32, 3)),
    MaxPooling2D((2, 2)),          # уменьшаем карту признаков
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(64, activation="relu"),
    Dense(10, activation="softmax"),
])

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.fit(X_train, y_train, epochs=5)

Слой Conv2D — это набор фильтров-рамок (32 и 64 фильтра в наших слоях), MaxPooling2D — уменьшение карты вдвое. Сравни точность этой CNN с обычной Dense-сетью на тех же картинках: CNN обычно заметно выигрывает именно потому, что учитывает расположение пикселей, а не рассматривает их как независимый список.

## Transfer learning: стоим на плечах гигантов

**Аналогия: готовый эксперт  💡**

Обучить мощную сеть с нуля — это месяцы вычислений и миллионы картинок, недоступные обычному человеку. Но есть готовые сети, уже обученные на гигантских наборах данных крупными компаниями. Transfer learning — это взять такого «эксперта», который уже прекрасно различает формы и текстуры, и доучить его под свою узкую задачу за пару минут. Как нанять опытного повара и показать ему пару своих рецептов, вместо того чтобы учить готовить с нуля.

Почему это работает? Нижние слои готовой сети уже научились находить универсальные детали (края, текстуры), которые нужны для распознавания чего угодно. Нам остаётся заменить только верхние слои под свои категории. Популярная готовая сеть — MobileNetV2: лёгкая, быстрая и точная, идеальна для учебных проектов. В следующей главе мы используем именно её, чтобы за один урок собрать классификатор собственных фотографий.

## Задания

Базовый уровень

Обучи CNN на CIFAR-10 (5 эпох). Затем собери обычную Dense-сеть (Flatten + Dense) на тех же данных и сравни точность на тесте. Насколько CNN лучше? Запиши обе цифры.

Со звёздочкой  ⭐

Добавь в CNN третий свёрточный блок (Conv2D + MaxPooling) или увеличь число фильтров. Как меняется точность и время обучения? Найди момент, когда усложнение перестаёт помогать, — это компромисс сложности и пользы.

Вопросы для обсуждения  💬

Почему обычная Dense-сеть теряет информацию о расположении пикселей, а CNN — нет?

Как иерархия признаков (кирпичи → дом) помогает распознавать сложные объекты?

Почему transfer learning экономит и время, и данные? В каких случаях он особенно полезен?

**Проверь себя  ℹ️**

1. Что делает свёрточный фильтр, скользя по картинке?

2. Зачем нужен слой пулинга?

3. Почему transfer learning работает — что уже «умеют» нижние слои готовой сети?